In [17]:

# 1---> Preprocessing And Cleaning
# 2--> Train Test Split
# 3---> Apply BOW ,TF-IDF ,Woed2Vec
# 4----> Apply machine leraning

In [18]:
# Data set laod
import pandas as pd

df = pd.read_csv("Amazon_Reviews.csv", encoding="latin1")
# df

In [19]:
# choosing rating and review test from the entire data sets 
data=df[["Rating","Review Text"]]
# data

In [20]:
# 1---> Preprocessing And Cleaning

# df.head()
data["Rating"].isnull().sum()
data.isnull().sum() 
data=data.dropna()       # handling missing value here
data.isnull().sum
# data["Rating"].value_counts()
# convert  text into numerical form of all the rating data
data["Rating"] = (
    data["Rating"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype("Int64")
)
# Assigned the value with neagative review is 0 and positive review is to be 1
data["Rating"]=data["Rating"].apply(lambda x: 0 if x<3 else 1 
                                   )
data["Rating"].value_counts()                                   

Rating
0    14350
1     6705
Name: count, dtype: int64

In [21]:
data["Rating"].value_counts()                                   

Rating
0    14350
1     6705
Name: count, dtype: int64

In [22]:
# Lowering the cases
data["Review Text"]=data["Review Text"].str.lower()

In [23]:
import nltk
from nltk.corpus import stopwords 
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
import re
from nltk.corpus import stopwords

# Load stopwords only ONCE
stop_words = set(stopwords.words("english"))

data["Review Text"] = (
    data["Review Text"]
    .fillna("")
    .astype(str)
    
    # Remove HTML tags
    .str.replace(r"<[^>]*>", " ", regex=True)
    
    # Remove URLs
    .str.replace(r"http\S+|www\S+", " ", regex=True)
    
    # Remove special characters
    .str.replace(r"[^a-zA-Z0-9\s]", " ", regex=True)
    
    # Remove extra spaces
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove stopwords
data["Review Text"] = data["Review Text"].apply(
    lambda x: " ".join(word for word in x.split() if word.lower() not in stop_words)
)

In [25]:
data = data.drop_duplicates(subset=["Review Text"], keep="first")
data


,Rating,Review Text
0,0,registered website tried order laptop entered ...
1,0,multiple orders one turned driver phone door n...
2,0,informed reprobates would going visit sick rel...
3,0,bought amazon problems happy service price ama...
4,0,could give lower rate would cancelled amazon p...
...,...,...
21209,1,perfect order fulfillment fast delivery amazon...
21210,1,perfect order fulfillment fast delivery amazon...
21211,1,always find going back amazon becouse prices g...
21212,1,placed abundance orders amazon last couple yea...


In [26]:
# !pip install nltk

In [27]:
# data["Review Text"]
# apply lemmatizer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
wnl=WordNetLemmatizer()
def lemmatize_words(text):
    words = word_tokenize(text)
    return " ".join([wnl.lemmatize(word) for word in words])
    

In [28]:
data["Review Text"] =data["Review Text"].apply(lemmatize_words)
# data
x=data["Review Text"]
y=data["Rating"]
x,y

(0        registered website tried order laptop entered ...
 1        multiple order one turned driver phone door nu...
 2        informed reprobate would going visit sick rela...
 3        bought amazon problem happy service price amaz...
 4        could give lower rate would cancelled amazon p...
                                ...                        
 21209    perfect order fulfillment fast delivery amazon...
 21210    perfect order fulfillment fast delivery amazon...
 21211    always find going back amazon becouse price go...
 21212    placed abundance order amazon last couple year...
 21213    good ordered amazon com delivered good order e...
 Name: Review Text, Length: 20376, dtype: object,
 0        0
 1        0
 2        0
 3        0
 4        0
         ..
 21209    1
 21210    1
 21211    1
 21212    1
 21213    1
 Name: Rating, Length: 20376, dtype: int64)

In [92]:
# train test split 
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)
# x_train.shape
x_train.shape

(16300,)

In [136]:
# Create a bag of words  ----> Text to vector 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

cv = CountVectorizer(
    max_features=1000,
    ngram_range=(1, 2)
)

X_train = cv.fit_transform(x_train).toarray()
X_test = cv.transform(x_test).toarray()

# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y = le.fit_transform(ds["labels"]) 

# spam_detect_models=MultinomialNB()
# spam_detect_models.fit(x, y)
# y

In [137]:
from sklearn.naive_bayes import GaussianNB
spam_model=GaussianNB()
spam_model.fit(X_train,y_train)

GaussianNB()

In [107]:
# X_test

In [138]:
y_pred=spam_model.predict(X_test)
y_pred,y_test

(array([1, 0, 0, ..., 1, 0, 1]),
 415      0
 1044     0
 6504     0
 8279     0
 16464    1
         ..
 8816     0
 4401     0
 8268     0
 5339     0
 16138    1
 Name: Rating, Length: 4076, dtype: int64)

In [135]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
confusion_matrix(y_test,y_pred)

0.7666830225711482
              precision    recall  f1-score   support

           0       0.92      0.72      0.81      2820
           1       0.58      0.87      0.70      1256

    accuracy                           0.77      4076
   macro avg       0.75      0.79      0.75      4076
weighted avg       0.82      0.77      0.78      4076



array([[2036,  784],
       [ 167, 1089]])